# CityFlo Bus Service — Metro Cities: Exploratory Data Analysis

**Dataset:** `cityflo_bus_service_metro_cities.csv` (3,258 rows × 36 columns)
**Workflow followed:** *Exploratory Data Analysis — A Standard 23-Step Checklist for Any Dataset*
(Classroom Computer Institute — Data Analytics)

This notebook walks through **all 23 steps** of the checklist, in order, across four phases:

| Phase | Steps | Goal |
|---|---|---|
| Phase 1 — Inspect | 1–8 | Understand the shape, structure, and quality of the raw data |
| Phase 2 — Clean & Prepare | 9–17 | Organize columns, clean values, export a final clean dataset |
| Phase 3 — Analyze | 18–21 | Explore relationships and statistically test them |
| Phase 4 — Report | 22–23 | Define KPIs and charts for the final dashboard |

Each step below has its own markdown explanation followed by the code that performs it.


In [ ]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from scipy.stats import gaussian_kde
import numpy as np

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", 50)

RAW_PATH = "/kaggle/input/datasets/rishijmanna/cityflo-bus-service-metro-cities-cleaned/cityflo_bus_service_metro_cities_cleaned.csv"
df = pd.read_csv(RAW_PATH)
df.head()

,trip_id,booking_id,customer_id,customer_name,gender,age,city,route_id,route_name,origin_stop,destination_stop,bus_number,bus_type,driver_id,driver_name,trip_date,scheduled_departure,actual_departure,scheduled_arrival,actual_arrival,distance_km,fare_inr,discount_inr,payment_mode,booking_channel,seat_number,trip_status,cancellation_reason,rating,occupancy_pct,weather,is_peak_hour,subscription_type,device_type,gps_enabled,complaint_raised
0,TRP000768,BK47984879,CUST01278,Shalini Verma,Female,21.0,Mumbai,RTMU005,Lower Parel - Mulund,Lower Parel,Mulund,MH23CF4441,AC Sleeper,DRV0080,Aarav Bhat,2024-09-21,20:25,20:25,20:58,20:58,10.9,153,0.0,UPI,Corporate Portal,D8,Completed,NaN,5.0,84.6,Haze,0,Corporate Plan,iOS,true,No
1,TRP002439,BK14257322,CUST01290,Kalpana Chopra,Male,24.0,Hyderabad,RTHY038,HITEC City - Kondapur,HITEC City,Kondapur,TS27CF8651,AC Seater,DRV0068,Sanjay Joshi,2024-08-08,14:50,15:12,15:29,15:51,14.1,127,0.0,Debit Card,Kiosk,D7,Delayed-Completed,NaN,NaN,28.3,Heavy Rain,False,Weekly Pass,Android,True,0
2,TRP000793,BK67236768,CUST01285,Sai Naidu,Female,18.0,Pune,RTPU012,Aundh - Magarpatta,Aundh,Magarpatta,MH24CF3646,AC Seater,DRV0001,Lakshmi Patel,2024-03-19,08:15,08:15,09:42,09:42,27.6,234,19.0,Cash,Corporate Portal,D11,Completed,NaN,3.0,15.4,Cloudy,1,Single Ride,Android,Yes,No
3,TRP002802,BK59659578,CUST00625,Priya Patel,Female,23.0,Mumbai,RTMU010,Ghatkopar - Lower Parel,Ghatkopar,Lower Parel,MH13CF5853,AC Sleeper,DRV0036,Siddharth Kumar,2024-12-29,17:45,17:45,18:57,NaN,28.6,310,0.0,Debit Card,Corporate Portal,D11,No-show,Customer Request,NaN,68.1,Clear,No,Corporate Plan,Web,1,Yes
4,TRP001653,BK19231279,CUST00276,Shalini Kulkarni,F,20.0,Bangalore,RTBA024,Hebbal - Electronic City,Hebbal,Electronic City,KA22CF6238,AC Seater,DRV0045,Sunita Gupta,2024-05-15,13:00,13:40,13:35,14:15,17.8,202,0.0,UPI,Corporate Portal,A1,Delayed-Completed,NaN,NaN,23.7,Haze,No,Corporate Plan,iOS,True,0


## Phase 4 — Report (Steps 22–23)

Define KPIs and charts for the final dashboard.

### Step 22 — Identify Key Metrics / KPIs for Dashboard
Chosen based on the problem statement (Step 12) and the Operations/Sales domain — each is
measurable, tied to the problem statement, and drives an actual decision:

- **Total Revenue** (`net_revenue_inr` summed)
- **Total Trips** and **Completed Trips**
- **Cancellation Rate %** (Cancelled + No-show ÷ total)
- **On-Time Performance %** (`delay_minutes` ≤ 5)
- **Average Fare** and **Average Occupancy %**
- **Average Rating**
- **Complaint Rate %**
- **Revenue by City** (identifies top/bottom performing metros)

In [192]:
kpis = {
    "Total Revenue (INR)": df["net_revenue_inr"].sum(),
    "Total Trips": len(df),
    "Completed Trips": (df["trip_status"].isin(["Completed", "Delayed-Completed"])).sum(),
    "Cancellation Rate %": (df["trip_status"].isin(["Cancelled", "No-show"]).mean() * 100),
    "On-Time Performance %": ((df["delay_minutes"] <= 5).mean() * 100),
    "Average Fare (INR)": df["fare_inr"].mean(),
    "Average Occupancy %": df["occupancy_pct"].mean(),
    "Average Rating": df["rating"].mean(),
    "Complaint Rate %": (df["complaint_raised"] == True).mean() * 100,
}
kpi_df = pd.DataFrame.from_dict(kpis, orient="index", columns=["value"]).round(2)
kpi_df

,value
Total Revenue (INR),859623.50
Total Trips,3200.00
Completed Trips,2494.00
Cancellation Rate %,22.06
On-Time Performance %,68.75
Average Fare (INR),317.68
Average Occupancy %,57.86
Average Rating,3.80
Complaint Rate %,6.41


In [172]:
df.groupby("city")["net_revenue_inr"].sum().sort_values(ascending=False).round(0)

city
Mumbai       157032.0
Pune         153194.0
Bangalore    148379.0
Chennai      140450.0
Hyderabad    135077.0
Delhi NCR    125492.0
Name: net_revenue_inr, dtype: float64

### Step 23 — Identify Important Charts for Dashboard
Chart types recommended by the checklist, applied to this dataset:

- **KPI cards** — the headline metrics from Step 22
- **Trend line chart** — revenue/trips over time (month)
- **Bar chart** — revenue by city
- **Pie/donut chart** — trip status split (already shown in Step 18)
- **Heatmap** — correlation matrix (already shown in Step 20) / weekday × city trip-density
- **Geo map** — not directly available (no lat/long in the data); city-level bar chart substitutes
- **Funnel chart** — booking → completed trip funnel
- **Scatter plot** — fare vs distance (already shown in Step 19)

In [193]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Monthly aggregation
monthly = df.groupby(df["trip_date"].dt.to_period("M")).agg(
    trips=("trip_id", "count"),
    revenue=("net_revenue_inr", "sum")
)

monthly.index = monthly.index.astype(str)

# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Revenue line
fig.add_trace(
    go.Scatter(
        x=monthly.index,
        y=monthly["revenue"],
        mode="lines+markers",
        name="Revenue (INR)",
        line=dict(color="#3b6fa0"),
        marker=dict(symbol="circle")
    ),
    secondary_y=False
)

# Trips line
fig.add_trace(
    go.Scatter(
        x=monthly.index,
        y=monthly["trips"],
        mode="lines+markers",
        name="Trips",
        line=dict(color="#c96a3f"),
        marker=dict(symbol="square")
    ),
    secondary_y=True
)

fig.update_layout(
    template="plotly_white",
    title="Monthly Revenue & Trip Volume Trend (2024)",
    title_x=0.5,
    xaxis_title="Month",
    height=500,
    width=1100,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

fig.update_xaxes(tickangle=45)

fig.update_yaxes(
    title_text="Revenue (INR)",
    title_font_color="#3b6fa0",
    tickfont_color="#3b6fa0",
    secondary_y=False
)

fig.update_yaxes(
    title_text="Trips",
    title_font_color="#c96a3f",
    tickfont_color="#c96a3f",
    secondary_y=True
)

fig.show(config={"responsive": True})

In [194]:
# Total revenue by city
revenue_city = (
    df.groupby("city")["net_revenue_inr"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)

fig = px.bar(
    revenue_city,
    x="city",
    y="net_revenue_inr",
    color_discrete_sequence=["#3b6fa0"],
    title="Total Revenue by City"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="City",
    yaxis_title="Revenue (INR)",
    xaxis_tickangle=30,
    showlegend=False,
    height=500,
    width=800
)

fig.show(config={"responsive": True})

In [195]:
# Create pivot table
heat_data = df.pivot_table(
    index="trip_weekday",
    columns="city",
    values="trip_id",
    aggfunc="count"
)

weekday_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

heat_data = heat_data.reindex(weekday_order)

fig = px.imshow(
    heat_data,
    text_auto=".0f",
    color_continuous_scale="Blues",
    aspect="auto",
    title="Trip Volume Heatmap — Weekday x City"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="City",
    yaxis_title="Weekday",
    height=500,
    width=900,
    coloraxis_colorbar_title="Trips"
)

fig.update_xaxes(side="bottom")

fig.show(config={"responsive": True})

In [196]:
# Funnel stages
funnel_stages = {
    "Booked": len(df),
    "Not Cancelled/No-show": (~df["trip_status"].isin(["Cancelled", "No-show"])).sum(),
    "Completed (on-time or delayed)": df["trip_status"].isin(
        ["Completed", "Delayed-Completed"]
    ).sum(),
    "Completed On-Time": (
        (df["trip_status"].isin(["Completed", "Delayed-Completed"])) &
        (df["delay_minutes"] <= 5)
    ).sum(),
}

funnel_df = pd.DataFrame({
    "stage": list(funnel_stages.keys()),
    "count": list(funnel_stages.values())
})

funnel_df["pct_of_booked"] = (
    funnel_df["count"] / funnel_df["count"].iloc[0] * 100
).round(1)

fig = px.bar(
    funnel_df,
    x="count",
    y="stage",
    orientation="h",
    text=funnel_df.apply(
        lambda r: f'{r["count"]} ({r["pct_of_booked"]}%)',
        axis=1
    ),
    color_discrete_sequence=["#3b6fa0"],
    title="Booking -> Completion Funnel"
)

fig.update_traces(
    textposition="outside",
    cliponaxis=False
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Count",
    yaxis_title="",
    yaxis=dict(autorange="reversed"),
    showlegend=False,
    height=450,
    width=800
)

fig.show(config={"responsive": True})

# Display dataframe
funnel_df

,stage,count,pct_of_booked
0,Booked,3200,100.0
1,Not Cancelled/No-show,2494,77.9
2,Completed (on-time or delayed),2494,77.9
3,Completed On-Time,1867,58.3


## Conclusion

All 23 steps of the checklist have been applied end-to-end: the raw dataset was inspected (Steps
1–8), cleaned and prepared into `cityflo_bus_service_metro_cities_cleaned.csv` (Steps 9–17),
statistically analyzed across univariate, bivariate, multivariate, and hypothesis-testing lenses
(Steps 18–21), and summarized into dashboard-ready KPIs and charts (Steps 22–23).

**Key findings to carry into a dashboard:**
- Fare scales strongly with distance (positive Pearson correlation).
- Bus type materially affects average fare (ANOVA significant).
- Delay behavior differs between peak and non-peak hours (t-test).
- Cancellation/no-show patterns are associated with city (Chi-square significant).
